PyTorch 自动求导 autograd

autograd 负责计算梯度，不负责决定怎样更新参数。优化器（如 `torch.optim.SGD`）读取参数的梯度并执行更新。



In [3]:
import torch
from torch import nn

3.1 `requires_grad`

In [2]:
# requires_grad=True 表示x这个张量要求计算梯度
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2

print(y)          # tensor(9., grad_fn=<PowBackward0>)
print(y.grad_fn)  # 类似 <PowBackward0 ...>

# 从结果向前遍历记录的运算，按链式法则计算导数
y.backward()
print(x.grad)     # tensor(6.)

tensor(9., grad_fn=<PowBackward0>)
tensor(6.)


In [10]:
# `requires_grad` 是是否追踪该张量运算的开关。只要运算中至少有一个输入需要梯度，且当前处于 grad mode，输出通常也会被追踪：
x = torch.tensor(3.0, requires_grad=True)
c = torch.tensor(2, dtype=torch.float) # 要计算梯度的张量必须是浮点数
y = c * x
print(y.grad_fn)
print(x.grad)
print(y.requires_grad)

# 反向传播计算梯度
y.backward()
print(y.grad_fn)
print(x.grad)


None
True
tensor(2.)


3.2 叶子张量、非叶子张量和 `grad_fn`

In [18]:
x = torch.tensor(3.0, requires_grad=True)
y = 3 * x
z = y + 1

# 叶子张量：通常由用户直接创建且requires_grad=True，不是由其他张量运算得到的。
# 非叶子张量：通常由其他张量运算得到，有对应的计算历史。
print(x.is_leaf, y.is_leaf, z.is_leaf)
# `grad_fn`：反向传播所需的运算节点信息，不是梯度本身。
print(x.grad_fn) # 张量没有导函数
print(y.grad_fn)
print(z.grad_fn)
# .grad：某个张量已计算出的梯度值。默认情况下，`.backward()` 会将梯度保存到需要梯度的 叶子 张量的 `.grad`；中间非叶子张量的 `.grad` 默认不保留。
z.backward()
print(x.grad)

True False False
None
tensor(3.)


In [9]:
# 如确实需要检查中间结果的梯度，可在反向传播前调用 `retain_grad()`：
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

# retain_grad 表示y这个非叶子节点要保留中间计算的梯度值
y.retain_grad()
z.backward()

print(x.grad)  # dz/dx = 2 * y * 3 = 36
print(y.grad)  # dz/dy = 2 * y = 12

tensor(36.)
tensor(12.)


3.3 `nn.Parameter` 与模型参数

`nn.Parameter` 是会被 `nn.Module` 自动注册的张量参数，默认 `requires_grad=True`：


In [25]:
# 3个输入特征，1个输出特征
layer = nn.Linear(3, 1)
for name, parameter in layer.named_parameters():
    print(name, parameter.shape, parameter.requires_grad, parameter.is_leaf)

weight torch.Size([1, 3]) True True
bias torch.Size([1]) True True


In [26]:
for name, parameter in layer.named_parameters():
    print(name, parameter.shape, parameter.requires_grad, parameter.is_leaf)

# 将参数计算梯度的能力冻结掉，此时被冻结的参数仍属于模型，但不再计算梯度：
for parameter in layer.parameters():
    parameter.requires_grad_(False)

for name, parameter in layer.named_parameters():
    print(name, parameter.shape, parameter.requires_grad, parameter.is_leaf)

weight torch.Size([1, 3]) True True
bias torch.Size([1]) True True
weight torch.Size([1, 3]) False True
bias torch.Size([1]) False True


4.1 标量损失调用 `.backward()`

In [10]:
x = torch.tensor(2.0, requires_grad=True)
loss = (x - 5) ** 2
# 反向传播计算梯度
loss.backward()
# 计算loss在x=2时的值
print(loss.item())

# 执行 loss.backward() 后，PyTorch 会把这个梯度保存到叶子张量 x 的 .grad 属性中。
# 计算loss在x=2出的导数
print(x.grad.item())
print(loss.grad) # 需要调用 loss.retain_grad()l

9.0
-6.0
None


/var/folders/62/z650ynp965s9rqn86r5kl6kr0000gn/T/ipykernel_18861/3704337926.py:11: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:493.)
  print(loss.grad) # 需要调用 loss.retain_grad()


In [30]:
# 4.2 非标量输出：向量-雅可比积
# 如果输出不是标量，PyTorch 不能凭空决定应该对多个输出如何求导。此时必须给 `.backward(gradient)` 提供与输出同形状的向量，它表示要计算的加权和的系数：

x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2
# y.backward() # RuntimeError: grad can be implicitly created only for scalar outputs
# 为向量计算梯度，必须先指定一个与其类似的向量, 当前使用全1向量
# 当y（即要计算梯度的变量）非标量时计算梯度，必须在backward里面传入一个与y相同的矩阵
# 这相当于告诉 PyTorch：对 y 中每个元素对应的输出，都使用 1 作为反向传播的初始梯度。
# 即告诉pytorch此时要为每一个变量x计算偏导数
y.backward(torch.ones_like(x)) # tensor([4., 6.])
print(x.grad)


tensor([4., 6.])


In [33]:
# 4.3 批次损失的 `sum` 和 `mean`
# 若逐样本损失为 `losses`：
# - `losses.sum()` 计算总和，梯度规模通常随批次大小增长。
# - `losses.mean()` 计算平均值，梯度按元素个数归一化。
# - `nn` 损失函数的 `reduction` 常可设为 `"none"`、`"sum"` 或 `"mean"`。

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y  = (3*x - 3) ** 2
y.backward(torch.ones_like(y))
print(y)
print(x.grad)
print(y.sum()) # sum = 0+9+36 = 45
print(y.mean()) # mean = sum/size = 45/3 = 15
print(x.sum())
print(x.mean())
print(x.grad.sum())
print(x.grad.mean())

tensor([ 0.,  9., 36.], grad_fn=<PowBackward0>)
tensor([ 0., 18., 36.])
tensor(45., grad_fn=<SumBackward0>)
tensor(15., grad_fn=<MeanBackward0>)
tensor(6., grad_fn=<SumBackward0>)
tensor(2., grad_fn=<MeanBackward0>)
tensor(54.)
tensor(18.)


In [40]:
 # 4.4 手动验证链式法则
x = torch.tensor(2.0, requires_grad=True)
u = x ** 2        # du/dx = 2x = 4
loss = 3 * u + 1  # dloss/du = 3
loss.backward()
print(x.grad)     # 3 * 4 = 12
# 计算图把各局部导数相乘：`dloss/dx = (dloss/du) * (du/dx)`。深层神经网络的反向传播正是将这个过程扩展到大量张量运算。

tensor(12.)


In [51]:
# 5. 梯度累积：最重要的训练习惯
# 调用 `.backward()` 时，梯度默认**累加**到 `.grad`，而不是覆盖旧值：
x = torch.tensor(3.0, requires_grad=True)
(x**2).backward() # 6
print(x.grad)
(2*x - 3).backward() # 6 + 2
print(x.grad)

# 这对于分批累积梯度有用，但一般每次参数更新前都需要清零：
# 每一个训练的epoch都必须清空梯度，

# x.grad = None 是只清空当前这一个特征x对应的梯度，在训练的时候一般是依赖优化器来清空所有特征的梯度，
# 类似：optimizer.zero_grad(set_to_none=True)，`set_to_none=True` 会把 `.grad` 设为 `None`，常可减少不必要的内存写入。
x.grad = None
(x ** 2).backward()
print(x.grad)


tensor(6.)
tensor(8.)
tensor(6.)


In [68]:
# 6. 标准训练循环：把概念串起来
model = nn.Linear(1, 1) # 模型
optimizer = torch.optim.SGD(model.parameters(), lr=0.1) # 优化器
criterion = nn.MSELoss() # 损失函数

# y = 2x+1
x = torch.tensor([[1.0], [2.0], [3.0], [4.0], [5.0]]) # 特征数据 x
target = torch.tensor([[3.0], [5.0], [7.0], [9.0], [11.0]]) # 目标标签 target
epochs = 20 # 训练轮次
for epoch in range(epochs):
    # 逐个样本更新，如果要训练一个批次，那么将x，target组成一个批次即可
    for step, (train_x, train_y) in enumerate(zip(x, target)):
        # 训练过程记忆顺序
        # 记忆顺序：**清梯度 → 前向 → 算损失 → 反向 → 更新**。`optimizer.step()` 不会替你调用 `backward()`；`backward()` 也不会修改参数值。
        optimizer.zero_grad(set_to_none=True)  # 1. 清理上一轮梯度
        prediction = model(train_x)            # 2. 前向计算，构建计算图
        loss = criterion(prediction, train_y)  # 3. 得到标量损失
        loss.backward()                        # 4. 反向传播，填充参数 .grad
        optimizer.step()                       # 5. 使用梯度更新参数

        if step % 2 == 0:
            print(epoch, step, loss.item())

print("训练后的权重:", model.weight.item())
print("训练后的偏置:", model.bias.item())

# 测试
with torch.no_grad():
    test_x = torch.tensor([[6.0]])
    print("x=4 时的预测值:", model(test_x).item())


0 0 10.329926490783691
0 2 0.051783572882413864
0 4 0.278719425201416
1 0 0.8219218254089355
1 2 0.016838716343045235
1 4 0.09063351154327393
2 0 0.267269492149353
2 2 0.005475417710840702
2 4 0.029470575973391533
3 0 0.08690692484378815
3 2 0.0017804852686822414
3 4 0.009582992643117905
4 0 0.028259893879294395
4 2 0.0005789874121546745
4 4 0.003116563195362687
5 0 0.009190237149596214
5 2 0.00018826554878614843
5 4 0.0010133811738342047
6 0 0.00298824580386281
6 2 6.121410115156323e-05
6 4 0.0003294344642199576
7 0 0.0009715324267745018
7 2 1.9903094653273e-05
7 4 0.00010708772606449202
8 0 0.0003158764448016882
8 2 6.4739870140329e-06
8 4 3.4859549487009645e-05
9 0 0.00010278941772412509
9 2 2.10682992474176e-06
9 4 1.1358821211615577e-05
10 0 3.346057565067895e-05
10 2 6.836571628809907e-07
10 4 3.6890887713525444e-06
11 0 1.0866089724004269e-05
11 2 2.2194944904185832e-07
11 4 1.190288458019495e-06
12 0 3.513551064315834e-06
12 2 7.207040653156582e-08
12 4 3.890054358635098e-07
13

In [75]:
# 7. 暂停梯度记录：训练、验证和推理
# torch.no_grad()： 通用性更强，适合验证、评估，以及需要在后续操作中继续使用张量的场景。冻结模型的一部分，继续训练其他部分（微调模型的部分参数）
# torch.inference_mode()： 限制更多，通常开销更低，适合纯推理、只需要预测结果的场景。
# model.eval()：切换到评估模式
# model.train()：切换到训练模式
# 建议是 训练模型评估的时候使用 no_grad，部署模型的时候使用 inference_mode

# 有时不需要构建计算图。例如验证或部署推理时，只计算输出而不更新参数，可使用梯度模式上下文。

# 7.1 `torch.no_grad()`， 训练模型评估时使用
# 开启模型的评估模式
model.eval()
# 不修改模型梯度
with torch.no_grad():
    # 块内的运算不会被 autograd 记录，结果通常 `requires_grad=False`。
    pred_x = model(torch.tensor([11.0]))
    print(pred_x)

# 7.2 `torch.inference_mode()`，部署推理模型时使用
model.eval()
with torch.inference_mode():
    pred_x = model(torch.tensor([11.0]))
    print(pred_x)


tensor([23.0001])
tensor([23.0001])


In [84]:
## 8. 冻结参数、分离计算图与张量复制
### 8.1 `detach()`：从当前图中分离


# detach() 的核心作用是：将一个张量从原来的自动求导计算图中分离出来，返回一个不再通过原计算图反向传播的张量。（创建一个新的与当前模型无关的新张量，类似于深拷贝一样）
# `detach()` 返回与当前图断开的张量，常用于将张量转成不参与本次求导的值。它通常与原张量共享底层存储，因此不要通过其中一个结果原地修改数据并期待另一个保持独立。确实需要独立副本时使用 `detach().clone()`。
# `tensor.detach().cpu().numpy()` 常用于导出结果到 NumPy；CUDA 张量需要先移到 CPU：
# array = prediction.detach().cpu().numpy()

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y_detached = y.detach()
print(y)
print(y_detached)
print(y.requires_grad)  # False
print(y_detached.requires_grad)  # False
print(y_detached.is_leaf)
print(y_detached.grad)
print(y_detached.grad_fn)


tensor(9., grad_fn=<PowBackward0>)
tensor(9.)
True
False
True
None
None
